In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import itertools

In [15]:
class memory(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.encoder = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.decoder = nn.RNN(vocab_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, vocab_size)

        ### use orthogonal initialization of weights ### 
        for name, param in self.encoder.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
        
        for name, param in self.decoder.named_parameters():
            if "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.zeros_(param)

    def forward(self, x):
        _, h = self.encoder(x)
        h_en = h.clone()
        dec_input = torch.zeros((1, 1, vocab_size))
        outputs = []
        for _ in range(seq_len):
            dec_out, h = self.decoder(dec_input, h)
            logits = self.out(dec_out)
            outputs.append(logits)
            dec_input = logits.detach()
        return torch.cat(outputs, dim=1), h_en.detach()

In [16]:
class prediction(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, context_size=0):
        super().__init__()
        self.context_size = 0
        self.linear1 = nn.Linear(input_size+context_size, hidden_size)
        self.linear2 = nn.Linear(hidden_size, output_size)

    def forward(self, h, context=None):
        if self.context_size == 0:
            out_ = F.relu(self.linear1(h))
        else:
            out_ = F.relu(self.linear1((h, context), dim=2))
        
        out = self.linear2(out_)

In [ ]:
# tokens start from 1
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-65
      
            self.y[ii] = \
                ord(data[ii+jj+1])-65

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [ ]:
def train_layer(model, optimizer, criterion, X, y, context=None, h=None, layer=0):
    if layer == 0:
        decoder_input = torch.cat([torch.zeros((1, 1), dtype=torch.long), X[:, :-1]], dim=1)
    else:
        decoder_input = torch.cat([torch.zeros((1, 1, X.shape[2]), dtype=torch.long), X[:, :-1]], dim=1)

    decoder_target = X.flip(1)

    optimizer.zero_grad()
    y_pred, decoder_output, h = model(X, decoder_input, context, h)
    # print(y_pred[0], 'y_pred', y[0], 'y', decoder_output[0], decoder_target[0])
    loss = criterion(y_pred[0], decoder_output[0], y[0], decoder_target[0])     
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if layer == 0:
        with torch.no_grad():
            if y[0] == y_pred[0].argmax():
                correct = 1
            else:
                correct = 0
        
        return correct, h.detach()
    
    return h.detach()
        

In [ ]:
total_samples = 10000
n_community = 2
n_members = 3

total_layers = 3
context_size = 30
hidden_size_memory = [20, 30, 40]
hidden_size_prediction = 50
vocab_size = n_community*n_members + 1
lr_memory = [1e-3,1e-3,1e-3,1e-3]
lr_prediction = 1e-3
short_term_memory = 3

memory_block = {}
prediction_block = {}
memory_criteria = []
memory_optimizers = []
prediction_criterion = nn.CrossEntropyLoss()

for layer in range(total_layers):
    if layer == 0:
        memory_block[layer] = memory(vocab_size, hidden_size_memory[layer])
        memory_criteria.append(
                nn.CrossEntropyLoss()
            )
    else:
        memory_block[layer] = memory(hidden_size_memory[layer-1], hidden_size_memory[layer])
        memory_criteria.append(
                nn.MSELoss()
            )
    
    if layer == 0:
        prediction_block[layer] = prediction(hidden_size_memory[layer], hidden_size_prediction, vocab_size, context_size=context_size)
    elif layer == total_layers-1:
        prediction_block[layer] = prediction(hidden_size_memory[layer], hidden_size_prediction, context_size)
    else:
        prediction_block[layer] = prediction(hidden_size_memory[layer], hidden_size_prediction, context_size, context_size=context_size)

    

    memory_optimizers.append(
        torch.optim.Adam(memory_block[layer].parameters(), lr=lr_memory[layer], weight_decay=1e-8)
    )

prediction_optimizer = torch.optim.Adam(
    itertools.chain(*[prediction_block[layer].parameters() for layer in range(total_layers)]),
    lr=lr_prediction,
    weight_decay=1e-8
)

### handle data ###
data = get_sequence(total_samples, n_community, n_member, train_percent=1.0)
data_set = Dataset_converter(data, 1, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

for X, y in train_loader:
        for layer in range(total_layers):
            # if total > 3000:
            #     downsample = downsample_factor**layer
            # else:
            #     downsample = 1
            downsample = downsample_factor**layer

            if layer == 0:
                accurate, h[layer] = train_layer(model[0], optimizers[0], criteria[0], X, y, context[0], h[layer])

                correct[total%1000] = accurate

            elif total%(downsample)==0:
                input_buffer[layer].append(
                    target_buffer[layer].clone()
                )
                target_buffer[layer] = h[layer-1][0][0]

                h[layer] = train_layer(
                    model[layer], optimizers[layer], criteria[layer], torch.stack(list(input_buffer[layer])).unsqueeze(0), 
                    torch.stack(list(target_buffer[layer])).unsqueeze(0), context[layer], h[layer], layer=layer
                )
                context[layer-1][0] = h[layer][0][0]
                 
        total += 1
        
        test_acc.append(
                np.sum(correct)/total if total<1000 else np.sum(correct)/1000
            )
            
        if total%1000 == 0:
            print(f'Iter : {total+1}, accuracy: {test_acc[-1]:.4f}')



In [18]:
memory_block

{0: memory(
   (encoder): RNN(7, 20, batch_first=True)
   (decoder): RNN(7, 20, batch_first=True)
   (out): Linear(in_features=20, out_features=7, bias=True)
 ),
 1: memory(
   (encoder): RNN(20, 30, batch_first=True)
   (decoder): RNN(20, 30, batch_first=True)
   (out): Linear(in_features=30, out_features=20, bias=True)
 ),
 2: memory(
   (encoder): RNN(30, 40, batch_first=True)
   (decoder): RNN(30, 40, batch_first=True)
   (out): Linear(in_features=40, out_features=30, bias=True)
 )}

In [19]:
prediction_block

{0: prediction(
   (linear1): Linear(in_features=50, out_features=50, bias=True)
   (linear2): Linear(in_features=50, out_features=7, bias=True)
 ),
 1: prediction(
   (linear1): Linear(in_features=60, out_features=50, bias=True)
   (linear2): Linear(in_features=50, out_features=30, bias=True)
 ),
 2: prediction(
   (linear1): Linear(in_features=40, out_features=50, bias=True)
   (linear2): Linear(in_features=50, out_features=30, bias=True)
 )}

In [21]:
memory_optimizers

[Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.001
     maximize: False
     weight_decay: 1e-08
 ),
 Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.001
     maximize: False
     weight_decay: 1e-08
 ),
 Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.001
     maximize: False
     weight_decay: 1e-08
 )]